In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
pd.set_option('display.max_rows', None)

In [2]:
train = pd.read_parquet('data/train.parquet')
test = pd.read_parquet('data/test.parquet')
val = pd.read_parquet('data/val.parquet')

In [3]:
x_cols = ['Attack ID', 'Avg source IP count', 'Detect count', 'Victim IP', 'Port number', 'Packet speed', 
          'Data speed', 'Avg packet len', 'Source IP count', 
          'Packet speed_normalized', 'Data speed_normalized', 'Avg packet len_normalized', 
          'total_seconds', 'weekday_number', 'time_of_day', 'IsWeekend', 'Start Hour', 'Sin_Hour', 
          'Cos_Hour', 'DayOfYear', 'Sin_DayOfYear', 'Cos_DayOfYear', 'Mean_DataSpeed', 'Std_DataSpeed', 
          'Min_DataSpeed', 'Max_DataSpeed', 'Mean_PacketSpeed', 'Min_PacketSpeed', 
          'Max_PacketSpeed', 'Mean_DetectCount', 'Std_DetectCount', 'Min_DetectCount', 'Max_DetectCount', 
          'VictimIP_Count', 'PortNumber_Count', 'AvgPacketLen_Mean', 'AvgPacketLen_Std', 
          'DataSpeed_PacketSpeed', 'PortFrequency', 'Std_DataSpeed_Replaced', 'Std_DetectCount_Replaced', 
          'AvgPacketLen_Std_Replaced', 'packet_Total', 'PacketSpeed_Per_Second',
          'DataSpeed_Per_TotalSeconds', 'AvgPacketLen_Per_TotalSeconds', 'Is_HTTP', 'Is_HTTPS', 
          'Is_FTP_Control', 'Is_FTP_Data', 'Is_SSH', 'Is_Telnet', 'Is_SMTP', 'Is_DNS', 'Is_POP3',
          'Is_IMAP', 'Is_DHCP', 'Is_SNMP', 'Is_LDAP', 'Is_LDAPS', 'Is_SMB_CIFS', 'Is_RDP', 'Is_SIP', 
          'Is_TFTP', 'Is_MySQL', 'Is_PostgreSQL', 'Is_Oracle', 'Is_HTTP_Alt_8080', 'Is_HTTP_Alt_8081',
          'Is_HTTP_Alt_80', 'Is_HTTPS_Alt_8443', 'Is_Syslog', 'Is_VNC', 'Is_IRC', 'Is_NTP', 'Is_Kerberos', 
          'Is_LDAP_Alt', 'Is_LDAPS_Alt', 'Is_RADIUS', 'Is_PPTP', 'Is_RTSP', 'Is_X11', 'Is_SNMP_Trap', 
          'Is_BGP', 'Is_IMAPS_Alt', 'Is_POP3S_Alt', 'Is_Telnet_SSL', 'Is_NNTP', 'Is_NNTPS', 'Is_LDAP_TLS', 
          'Is_AFS', 'Is_NFS', 'Is_SOCKS', 'Is_RSYNC', 'Is_CUPS', 'Is_TFTP_Alt', 'Is_Modbus', 'Is_CoAP', 
          'Is_MQTT', 'Is_AMQP', 'Is_Redis', 'Is_Memcached', 'Is_Elasticsearch', 'Is_Zookeeper', 
          'Is_Cassandra', 'Is_Docker', 'Is_Kubernetes', 'Is_SMB_Direct', 'Is_iSCSI', 'Is_AFP', 
          'Is_DHCPv6', 'Is_RIPng', 'Is_OSPF', 'Is_PPPoE', 'Is_L2TP', 'Is_GRE', 'Is_ESP', 'Is_AH',
          'PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5']

In [ ]:
import autosklearn.classification as ask
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Optional: use a subsample to reduce runtime / memory. Set to 1.0 to use full training set.
sample_frac = 1.0

# Prepare data
features = x_cols if 'x_cols' in globals() else train.select_dtypes(include=[np.number]).columns.drop('Type', errors='ignore').tolist()
if sample_frac < 1.0:
    train_sample = train.sample(frac=sample_frac, random_state=42)
else:
    train_sample = train

X = train_sample[features]
y = train_sample['Type'].astype(int)

X_test = test[features]
y_test = test['Type'].astype(int)

# Split a holdout for quick validation inside the notebook (optional)
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# Configure and run Auto-sklearn
automl = ask.AutoSklearnClassifier(
    time_left_for_this_task=300,       # total seconds for the search (adjust)
    per_run_time_limit=60,            # per model-fitting time limit
    tmp_folder="autosklearn_tmp",
    output_folder="autosklearn_out",
    n_jobs=1,                         # keep 1 to avoid excessive memory use
    seed=42,
    resampling_strategy='cv',
    resampling_strategy_arguments={'folds': 5}
)

automl.fit(X_train.copy(), y_train.copy(), dataset_name="ddos_auto")

# Quick summary
try:
    print(automl.sprint_statistics())
    print("Top models:")
    automl.show_models()
except Exception:
    pass

# Evaluate on holdout and test
y_pred_hold = automl.predict(X_hold)
print("Holdout - Accuracy:", accuracy_score(y_hold, y_pred_hold))
print("Holdout - F1 (weighted):", f1_score(y_hold, y_pred_hold, average='weighted', zero_division=0))
print(classification_report(y_hold, y_pred_hold, zero_division=0))

y_pred_test = automl.predict(X_test)
print("Test  - Accuracy:", accuracy_score(y_test, y_pred_test))
print("Test  - F1 (weighted):", f1_score(y_test, y_pred_test, average='weighted', zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, zero_division=0))
# ...existing code...